In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    GRU,
    Dense,
    Dropout,
    MultiHeadAttention,
    LayerNormalization,
    GlobalAveragePooling1D
)

print("TensorFlow:", tf.__version__)
print("All imports successful!")

TensorFlow: 2.21.0
All imports successful!


In [3]:
# ============================================
# CELL 2 — LOAD DATA
# ============================================

import pandas as pd
import numpy as np

file_path = "worldbank_inflation_model_ready.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (240, 70)

Columns:
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

First 5 rows:


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,...,-0.931196,-1.028282,3.626041,4.257462,4.257462,4.257462,4.257462,4.257462,4.257462,4.257462
1,Africa Eastern and Southern,AFE,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,19.598394,19.598394,19.598394,19.598394,19.598394,19.598394,...,6.446877,6.221375,4.689806,4.102851,5.191629,6.824727,10.883478,7.399186,4.770857,4.270111
2,Afghanistan,AFG,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,12.686269,12.686269,12.686269,12.686269,12.686269,12.686269,...,4.383892,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,-6.601186
3,Africa Western and Central,AFW,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,8.799211,8.799211,8.799211,8.799211,8.799211,8.799211,...,1.487416,1.725486,1.784050,1.983092,2.490378,3.745568,7.949251,5.221168,3.608044,1.657818
4,Angola,AGO,"Inflation, consumer prices (annual %)",FP.CPI.TOTL.ZG,83.783784,83.783784,83.783784,83.783784,83.783784,83.783784,...,30.694415,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495,20.162020


In [4]:
# ============================================
# CELL 3 — YEAR COLUMNS
# ============================================

START_YEAR = 1960
END_YEAR = 2025

year_columns = [str(year) for year in range(START_YEAR, END_YEAR + 1)]

print("Number of year columns:", len(year_columns))
print("First year:", year_columns[0])
print("Last year:", year_columns[-1])

# Check missing year columns
missing_years = [year for year in year_columns if year not in df.columns]

print("\nMissing year columns:", missing_years)

Number of year columns: 66
First year: 1960
Last year: 2025

Missing year columns: []


In [5]:
# ============================================
# CELL 4 — SELECT REQUIRED COLUMNS
# ============================================

id_columns = ["Country Name", "Country Code"]

feature_df = df[id_columns + year_columns].copy()

print("Feature dataset shape:", feature_df.shape)

display(feature_df.head())

Feature dataset shape: (240, 68)


,Country Name,Country Code,1960,1961,1962,1963,1964,1965,1966,1967,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,4.032258,...,-0.931196,-1.028282,3.626041,4.257462,4.257462,4.257462,4.257462,4.257462,4.257462,4.257462
1,Africa Eastern and Southern,AFE,19.598394,19.598394,19.598394,19.598394,19.598394,19.598394,19.598394,19.598394,...,6.446877,6.221375,4.689806,4.102851,5.191629,6.824727,10.883478,7.399186,4.770857,4.270111
2,Afghanistan,AFG,12.686269,12.686269,12.686269,12.686269,12.686269,12.686269,12.686269,12.686269,...,4.383892,4.975952,0.626149,2.302373,5.601888,5.133203,13.712102,-4.644709,-6.601186,-6.601186
3,Africa Western and Central,AFW,8.799211,8.799211,8.799211,8.799211,8.799211,8.799211,8.799211,8.799211,...,1.487416,1.725486,1.784050,1.983092,2.490378,3.745568,7.949251,5.221168,3.608044,1.657818
4,Angola,AGO,83.783784,83.783784,83.783784,83.783784,83.783784,83.783784,83.783784,83.783784,...,30.694415,29.844480,19.628938,17.080954,22.271539,25.754295,21.355290,13.644102,28.240495,20.162020


In [7]:
# ============================================
# CELL 4 — CREATE THE 193-COUNTRY DATASET
# ============================================

# World Bank aggregate/non-country codes to remove
aggregate_codes = [
    "AFE", "AFW", "ARB", "CEB", "CSS",
    "EAP", "EAR", "EAS", "ECA", "ECS",
    "EMU", "EUU", "FCS", "HIC", "HPC",
    "IBD", "IBT", "IDA", "IDB", "IDX",
    "INX", "LAC", "LCN", "LDC", "LIC",
    "LMC", "LMY", "LTE", "MEA", "MIC",
    "MNA", "NAC", "OED", "OSS", "PRE",
    "PSS", "PST", "SAS", "SSA", "SSF",
    "SST", "TEA", "TEC", "TLA", "TMN",
    "TSA", "TSS", "UMC", "WLD"
]

# Remove aggregate entries
feature_df = df[
    ~df["Country Code"].isin(aggregate_codes)
][["Country Name", "Country Code"] + year_columns].copy()

# Remove rows that don't have usable inflation data
feature_df = feature_df.dropna(
    subset=year_columns,
    how="all"
)

# Reset index
feature_df.reset_index(drop=True, inplace=True)

print("Feature dataset shape:", feature_df.shape)

print("\nNumber of countries:", len(feature_df))

print("\nFirst 10 countries:")
display(feature_df[["Country Name", "Country Code"]].head(10))

Feature dataset shape: (193, 68)

Number of countries: 193

First 10 countries:


,Country Name,Country Code
0,Aruba,ABW
1,Afghanistan,AFG
2,Angola,AGO
3,Albania,ALB
4,United Arab Emirates,ARE
5,Argentina,ARG
6,Armenia,ARM
7,Antigua and Barbuda,ATG
8,Australia,AUS
9,Austria,AUT


In [8]:
# ============================================
# CELL 5 — MISSING VALUE CHECK
# ============================================

feature_df[year_columns] = feature_df[year_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

print("Total missing values:",
      feature_df[year_columns].isna().sum().sum())

print("\nMissing values by year:")
display(
    feature_df[year_columns].isna().sum()
)

Total missing values: 0

Missing values by year:


1960    0
1961    0
1962    0
1963    0
1964    0
       ..
2021    0
2022    0
2023    0
2024    0
2025    0
Length: 66, dtype: int64

In [9]:
# ============================================
# CELL 6 — STATISTICAL FEATURES
# ============================================

feature_df["Mean_Inflation"] = feature_df[year_columns].mean(axis=1)

feature_df["Median_Inflation"] = feature_df[year_columns].median(axis=1)

feature_df["Std_Inflation"] = feature_df[year_columns].std(axis=1)

feature_df["Min_Inflation"] = feature_df[year_columns].min(axis=1)

feature_df["Max_Inflation"] = feature_df[year_columns].max(axis=1)

feature_df["Inflation_Range"] = (
    feature_df["Max_Inflation"]
    - feature_df["Min_Inflation"]
)

display(
    feature_df[
        [
            "Country Name",
            "Country Code",
            "Mean_Inflation",
            "Median_Inflation",
            "Std_Inflation",
            "Min_Inflation",
            "Max_Inflation",
            "Inflation_Range"
        ]
    ].head(10)
)

,Country Name,Country Code,Mean_Inflation,Median_Inflation,Std_Inflation,Min_Inflation,Max_Inflation,Inflation_Range
0,Aruba,ABW,3.482729,4.032258,1.813742,-2.372065,8.955987,11.328052
1,Afghanistan,AFG,10.141689,12.686269,5.747544,-6.811161,26.418664,33.229825
2,Angola,AGO,208.945207,83.783784,617.788703,7.280387,4145.105982,4137.825595
3,Albania,ALB,116.790690,155.505086,110.598079,0.050018,226.005421,225.955403
4,United Arab Emirates,ARE,9.460938,12.250420,4.855784,-2.079403,12.250420,14.329823
5,Argentina,ARG,42.606311,34.277224,34.250358,34.277224,219.883929,185.606705
6,Armenia,ARM,1793.816644,3373.759443,1691.779274,-1.403608,3373.759443,3375.163051
7,Antigua and Barbuda,ATG,1.580327,1.121288,1.331486,-0.550160,7.531078,8.081238
8,Australia,AUS,4.652285,3.286049,3.664040,-0.319489,15.416677,15.736166
9,Austria,AUT,3.365980,2.852291,2.098407,0.506309,9.521788,9.015479


In [10]:
# ============================================
# CELL 7 — RECENT INFLATION FEATURES
# ============================================

recent_5_years = [str(y) for y in range(2021, 2026)]
recent_10_years = [str(y) for y in range(2016, 2026)]
previous_10_years = [str(y) for y in range(2006, 2016)]

feature_df["Mean_Recent_5Y"] = (
    feature_df[recent_5_years].mean(axis=1)
)

feature_df["Mean_Recent_10Y"] = (
    feature_df[recent_10_years].mean(axis=1)
)

feature_df["Mean_Previous_10Y"] = (
    feature_df[previous_10_years].mean(axis=1)
)

display(
    feature_df[
        [
            "Country Name",
            "Country Code",
            "Mean_Recent_5Y",
            "Mean_Recent_10Y",
            "Mean_Previous_10Y"
        ]
    ].head(10)
)

,Country Name,Country Code,Mean_Recent_5Y,Mean_Recent_10Y,Mean_Previous_10Y
0,Aruba,ABW,4.257462,3.146880,2.136720
1,Afghanistan,AFG,0.199645,1.888848,6.689467
2,Angola,AGO,21.831240,22.867653,11.540702
3,Albania,ALB,3.577416,2.620921,2.543763
4,United Arab Emirates,ARE,2.002420,1.265456,4.824803
5,Argentina,ARG,138.819386,89.249200,34.277224
6,Armenia,ARM,4.276522,2.612367,5.054349
7,Antigua and Barbuda,ATG,4.445763,2.743637,2.130907
8,Australia,AUS,4.219126,2.869034,2.643547
9,Austria,AUT,5.118556,3.347683,1.942068


In [11]:
# ============================================
# CELL 8 — LONG-TERM TREND
# ============================================

feature_df["Long_Term_Change"] = (
    feature_df["2025"] - feature_df["1960"]
)

feature_df["Long_Term_Trend"] = (
    feature_df["2025"] - feature_df["1960"]
) / 65

display(
    feature_df[
        [
            "Country Name",
            "Country Code",
            "Long_Term_Change",
            "Long_Term_Trend"
        ]
    ].head(10)
)

,Country Name,Country Code,Long_Term_Change,Long_Term_Trend
0,Aruba,ABW,0.225204,0.003465
1,Afghanistan,AFG,-19.287454,-0.296730
2,Angola,AGO,-63.621764,-0.978796
3,Albania,ALB,-223.859236,-3.443988
4,United Arab Emirates,ARE,-10.999555,-0.169224
5,Argentina,ARG,185.606705,2.855488
6,Armenia,ARM,-3370.452510,-51.853116
7,Antigua and Barbuda,ATG,0.247447,0.003807
8,Australia,AUS,-0.854784,-0.013151
9,Austria,AUT,1.581445,0.024330


In [12]:
# ============================================
# CELL 9 — VOLATILITY
# ============================================

feature_df["Inflation_Volatility"] = (
    feature_df[year_columns].std(axis=1)
)

display(
    feature_df[
        [
            "Country Name",
            "Country Code",
            "Inflation_Volatility"
        ]
    ].head(10)
)

,Country Name,Country Code,Inflation_Volatility
0,Aruba,ABW,1.813742
1,Afghanistan,AFG,5.747544
2,Angola,AGO,617.788703
3,Albania,ALB,110.598079
4,United Arab Emirates,ARE,4.855784
5,Argentina,ARG,34.250358
6,Armenia,ARM,1691.779274
7,Antigua and Barbuda,ATG,1.331486
8,Australia,AUS,3.664040
9,Austria,AUT,2.098407


In [13]:
# ============================================
# CELL 10 — INFLATION SHOCK FEATURES
# ============================================

feature_df["High_Inflation_Years"] = (
    feature_df[year_columns] > 10
).sum(axis=1)

feature_df["Very_High_Inflation_Years"] = (
    feature_df[year_columns] > 20
).sum(axis=1)

feature_df["Negative_Inflation_Years"] = (
    feature_df[year_columns] < 0
).sum(axis=1)

display(
    feature_df[
        [
            "Country Name",
            "Country Code",
            "High_Inflation_Years",
            "Very_High_Inflation_Years",
            "Negative_Inflation_Years"
        ]
    ].head(10)
)

,Country Name,Country Code,High_Inflation_Years,Very_High_Inflation_Years,Negative_Inflation_Years
0,Aruba,ABW,0,0,4
1,Afghanistan,AFG,49,1,5
2,Angola,AGO,63,53,0
3,Albania,ALB,38,37,0
4,United Arab Emirates,ARE,49,0,2
5,Argentina,ARG,66,66,0
6,Armenia,ARM,38,36,2
7,Antigua and Barbuda,ATG,0,0,2
8,Australia,AUS,7,0,1
9,Austria,AUT,0,0,0


In [14]:
# ============================================
# CELL 11 — YEAR-TO-YEAR CHANGES
# ============================================

inflation_diff = feature_df[year_columns].diff(
    axis=1
)

# Remove the 1960 column because it has no previous year
inflation_diff = inflation_diff.iloc[:, 1:]

diff_columns = [
    f"Diff_{year}"
    for year in range(1961, 2026)
]

inflation_diff.columns = diff_columns

feature_df = pd.concat(
    [feature_df, inflation_diff],
    axis=1
)

print("Year-to-year difference features added.")

print("New dataset shape:", feature_df.shape)

Year-to-year difference features added.
New dataset shape: (193, 148)


In [15]:
# ============================================
# CELL 12 — RECENT TRENDS
# ============================================

feature_df["Trend_2016_2025"] = (
    feature_df["2025"] - feature_df["2016"]
) / 9

feature_df["Trend_2020_2025"] = (
    feature_df["2025"] - feature_df["2020"]
) / 5

feature_df["Trend_2021_2025"] = (
    feature_df["2025"] - feature_df["2021"]
) / 4

display(
    feature_df[
        [
            "Country Name",
            "Country Code",
            "Trend_2016_2025",
            "Trend_2020_2025",
            "Trend_2021_2025"
        ]
    ].head(10)
)

C:\Users\Aarjav\AppData\Local\Temp\ipykernel_11644\1709855892.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_df["Trend_2016_2025"] = (
C:\Users\Aarjav\AppData\Local\Temp\ipykernel_11644\1709855892.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_df["Trend_2020_2025"] = (
C:\Users\Aarjav\AppData\Local\Temp\ipykernel_11644\1709855892.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all c

,Country Name,Country Code,Trend_2016_2025,Trend_2020_2025,Trend_2021_2025
0,Aruba,ABW,0.576518,0.000000,0.000000
1,Afghanistan,AFG,-1.220564,-2.440615,-2.933597
2,Angola,AGO,-1.170266,-0.421904,-1.398069
3,Albania,ALB,0.096750,0.105060,0.026178
4,United Arab Emirates,ARE,-0.040736,0.666054,0.267732
5,Argentina,ARG,20.622967,35.573767,42.868638
6,Armenia,ARM,0.523393,0.419100,-0.969476
7,Antigua and Barbuda,ATG,0.206464,0.148549,-0.173565
8,Australia,AUS,0.177449,0.405427,0.002531
9,Austria,AUT,0.292845,0.429057,0.190132


In [16]:
# Defragment the DataFrame
feature_df = feature_df.copy()

print("DataFrame defragmented successfully.")
print("Current shape:", feature_df.shape)

DataFrame defragmented successfully.
Current shape: (193, 151)


In [17]:
# ============================================
# CELL 13: FEATURE ENGINEERING SUMMARY
# ============================================

print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print("\nFeature dataset shape:")
print(feature_df.shape)

print("\nNumber of countries:")
print(feature_df["Country Code"].nunique())

print("\nNumber of features:")
print(feature_df.shape[1] - 2)

print("\nMissing values:")
print(feature_df.isna().sum().sum())

print("\nInfinite values:")
print(np.isinf(
    feature_df.select_dtypes(include=np.number)
).sum().sum())

print("\nFeature columns:")
print(feature_df.columns.tolist())

FEATURE ENGINEERING SUMMARY

Feature dataset shape:
(193, 151)

Number of countries:
193

Number of features:
149

Missing values:
0

Infinite values:
0

Feature columns:
['Country Name', 'Country Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Mean_Inflation', 'Median_Inflation', 'Std_Inflation', 'Min_Inflation', 'Max_Inflation', 'Inflation_Range', 'Mean_Recent_5Y', 'Mean_Recent_10Y', 'Mean_Previous_10Y', 'Long_Term_Change', 'Long_Term_Trend', 'Inflation_Volatility', 'High_Inflation_Years', 'Very_High_Infl

In [18]:
# ============================================
# CELL 14: DATA QUALITY CHECK
# ============================================

numeric_cols = feature_df.select_dtypes(include=np.number).columns

print("NaN values:", feature_df[numeric_cols].isna().sum().sum())
print("Infinite values:", np.isinf(feature_df[numeric_cols]).sum().sum())

# Replace infinity if any exists
feature_df[numeric_cols] = feature_df[numeric_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

# Fill any remaining NaN values with column median
for col in numeric_cols:
    if feature_df[col].isna().any():
        feature_df[col] = feature_df[col].fillna(feature_df[col].median())

print("\nAfter cleaning:")
print("NaN values:", feature_df[numeric_cols].isna().sum().sum())
print("Infinite values:", np.isinf(feature_df[numeric_cols]).sum().sum())

NaN values: 0
Infinite values: 0

After cleaning:
NaN values: 0
Infinite values: 0


In [19]:
# ============================================
# CELL 15: SAVE FEATURE DATASET
# ============================================

feature_file = "inflation_feature_engineered.csv"

feature_df.to_csv(feature_file, index=False)

print("Feature-engineered dataset saved:")
print(feature_file)

print("Final shape:", feature_df.shape)

Feature-engineered dataset saved:
inflation_feature_engineered.csv
Final shape: (193, 151)
